In [1]:
## Load libraries
import pandas as pd
import numpy as np
import sys
import matplotlib.pyplot as plt
import matplotlib.cm as cm
from keras.datasets import  mnist
plt.style.use('dark_background')
%matplotlib inline

In [2]:
np.set_printoptions(precision=2)
import tensorflow as tf
tf.__version__

'2.14.0'

In [3]:
#Load MNIST data
(X_train, y_train), (X_test, y_test) = mnist.load_data()
X_train = X_train.transpose(1, 2, 0)
X_test = X_test.transpose(1, 2, 0)
X_train = X_train.reshape(X_train.shape[0] * X_train.shape[1], X_train.shape[2])
X_test = X_test.reshape(X_test.shape[0] * X_test.shape[1], X_test.shape[2])


In [4]:
num_labels = len(np.unique(y_train))
num_features = X_train.shape[0]
num_samples = X_train.shape[1]

In [7]:
#One hot encode class labels
Y_train = tf.keras.utils.to_categorical(y_train).T
Y_test = tf.keras.utils.to_categorical(y_test).T


In [8]:
#Normalizze the samples (images)
xmax = np.amax(X_train)
xmin = np.amin(X_train)
X_train = (X_train - xmin)/(xmax - xmin) #all train features turn into a number between 1 and 0
X_test = (X_test - xmin)/(xmax - xmin) 

In [9]:
print('MNIST set')
print('----------')
print('Number of training samples = %d'%(num_samples))
print('Number of features = %d'%(num_features))
print('Number of output classes = %d'%(num_labels))

MNIST set
----------
Number of training samples = 60000
Number of features = 784
Number of output classes = 10


In [10]:
class Layer:
  def __init__(self):
    self.input = None
    self.output = None

  def forward(self, input):
    pass

  def backward(self, output_gradient, learning_rate):
    pass

In [18]:
## Define the loss function and its gradient
def cce(Y, Yhat):
  return(np.mean(np.sum(-Y*np.log(Yhat), axis=0)))

def cce_gradient(Y, Yhat):
  return(-Y/Yhat)

# TensorFlow in-built function for categorical crossentropy loss
#cce = tf.keras.losses.CategoricalCrossentropy()

In [12]:
## Softmax activation layer class
class Softmax(Layer):
  def forward(self, input):
    self.output = tf.nn.softmax(input, axis=0).numpy()

  def backward(self, output_gradient, learning_rate = None):
    ## Following is the inefficient way of calculating the backward gradient
    softmax_gradient =  np.empty((self.input.shape[0], output_gradient.shape[1]), dtype=np.float64)
    for b in range(softmax_gradient.shape[1]):
      softmax_gradient[:, b] = np.dot((np.identity(self.output.shape[0]) - self.output[:, b].T) * self.output[:, b], output_gradient[:, b] )
    return softmax_gradient
  
  ## Following is the efficient of calculating the backward gradient
  #T = (np.transpose(np.identity(self.output.shape[0]) - np.atleast_2d(self.output).T[:, np.newaxis, :], (1, 2, 0)) * np.atleast_2d(se
  #return(np.einsum('ijk, ik -> jk', T, output_gradient))


In [20]:
## Dense layer class
class Dense(Layer):
    def __init__(self, input_size, output_size):
        self.weights = 0.01 * np.random.randn(output_size, input_size + 1) # +1 for bias trick
        self.weights[:, -1] = 0.01 # Set all bias values to the same nonzero constant

    def forward(self, input):
        self.input = np.vstack([input, np.ones((1, input.shape[1]))]) #bias trick
        self.output= np.dot(self.weights, self.input)

    def backward(self, output_gradient, learning_rate):
        ## Following is the inefficient way of calculating the backward gradient
        dense_gradient = np.zeros((self.output.shape[0], self.output.shape[1]), dtype=np.float64)
        for b in range(output_gradient.shape[1]):
          dense_gradient += np.dot(output_gradient[:, b].reshape(-1, 1), self.input[:, b].reshape(-1, 1).T)
        dense_gradient =(1/output_gradient.shape[1])*dense_gradient

        ## Following is the efficient way of calculating the backward gradient
        #dense_gradient = (1/output_gradient.shape[1])1])*np.dot(np.atleast_2d(output_gradient), np.atleast_2d(self.input).T)
        #self.weights = learning_rate * (-dense_gradient)

In [21]:
#Function to generate sample indices for batch processing according to batch size
def generate_batch_indices(num_samples, batch_size):
    #reorder sample indices
    reordered_sample_indices= np.random.choice(num_samples, num_samples, replace=False)
    #Generate batch indices for batch processing
    batch_indices = np.split(reordered_sample_indices, np.arange(batch_size, len(reordered_sample_indices), batch_size))
    return batch_indices

In [22]:
#Example generation of batch indices
batch_size = 100
batch_indices = generate_batch_indices(num_samples, batch_size)
print(batch_indices)

[array([50064, 21861, 16336, 18623, 56150, 59685, 16591,  6132, 12833,
       45562, 14104, 45795, 10792, 15557, 16072,  3873, 43651, 22145,
       16130, 12279,  7670, 43749, 41454, 52642, 19838, 20033, 27945,
       20247, 20123, 30510,  8016, 54076, 52841,  9694, 40360,  5802,
        7375, 11505, 46987,  5469, 58481, 16159, 38591,  6888, 44329,
       38990, 59977, 46043, 15274, 13753, 24859,  6444, 14476, 45566,
       46133, 42238, 33026,  6046, 21153, 22770, 31370, 27992, 49799,
       26242, 15061, 33389, 51675, 27865, 15752, 34863, 12992, 11004,
       36306, 25944, 25726, 34181, 32141,     5, 59962, 26795, 38593,
       50110, 33455, 11688, 25467, 46652, 36181, 36313, 56825, 59961,
       34099, 59764, 27223, 30080, 32910, 46013, 13481, 26785, 40552,
       11077]), array([ 3130, 48264,  6749, 44645, 27601, 59347, 49289, 44959, 41814,
       17209,  4407, 43310, 20311, 10198, 42492, 35863,  6258, 48247,
       20514, 38117, 47806, 45411, 36781, 23735,   919, 17657,  3680,
   

In [23]:
#Train tge 0-layer neural network using batch training with batch size = 16
learning_rate = 0.1
batch_size = 16
nepochs = 100
loss_epoch = np.empty(nepochs,dtype=np.float64) #create empty array to store loss values for each epoch


In [25]:
#Neural network architecture
dlayer = Dense(num_features, num_labels)
softmax = Softmax()

In [26]:
# Steps: Run over each sample in the bacth, ca;culate loss, gradient of loss, update weights

epoch = 0
while epoch < nepochs:
    batch_indices = generate_batch_indices(num_samples, batch_size)
    loss = 0
    for b in range(len(batch_indices)):
        dlayer.forward(X_train[:, batch_indices[b]])
        softmax.forward(dlayer.output)
        loss += cce(Y_train[:, batch_indices[b]], softmax.output)

        #backward prop starts here
        grad = cce_gradient
        grad = softmax.backward(grad)
        grad = dlayer.backward(grad, learning_rate)

    loss_epoch[epoch] = loss/len(batch_indices)
    print('Epoch %d: loss = %f'%(epoch+1, loss_epoch[epoch]))
    epoch = epoch + 1
    


AttributeError: 'NoneType' object has no attribute 'shape'